# Phasor Equilibrium Propagation

This notebook adapts vanilla equilibrium propagation (EP) to a
**phase-based** network. Each neuron's state is a phase
$\theta \in [-1, 1]$ (units of $\pi$), equivalently a unit complex
$z = e^{i\pi\theta}$. We sidestep the holomorphic-activation
constraint that limits hEP — Liouville's theorem says no globally
saturating holomorphic activation exists — by encoding the
"saturation" geometrically as the **unit-circle constraint**.

See `docs/phasor_ep_design.md` for the design background. This
notebook is the simplest first cut: discrete-time settling, two
real-valued layers, single training example, with the same gradient
verification pattern as `ep_demo.ipynb`.

Sections:
1. State, energy, and dynamics
2. Free-phase settling (with magnitude + angle plots)
3. Nudged phase and the EP gradient
4. Gradient verification vs. finite differences (cosine, rel-err, rel-mag-err)
5. A simple training run


In [ ]:
using Pkg; Pkg.activate("..")
using LinearAlgebra, Statistics, Random, Plots
using Random: Xoshiro


## 0. Using the package API

The package now ships a vanilla EP module (`src/ep.jl`) that wires the
prototype's algorithm into the existing `PhasorDense` layer. You can
build any `Lux.Chain` of `PhasorDense` layers (`use_bias=false` in
Phase 1) and train it with one call:

```julia
chain = Chain(PhasorDense(n_in => n_hid, normalize_to_unit_circle, use_bias=false),
              PhasorDense(n_hid => n_out, normalize_to_unit_circle, use_bias=false))
ps, st = Lux.setup(rng, chain)
losses, ps, st = ep_train(chain, ps, st, train_loader, args; method=StaticEP(β=0.1))
```

The cell below does this end-to-end on the same fixed-pattern task as
the inline prototype below in Sections 1–5. The rest of the notebook
walks through the underlying mechanics step by step (`PhasorEPNet`
struct, energy function, settling, FD verification, training loop,
lock-in detection (temporal real probe)) — useful both as documentation
and as the reference implementation that the package API was tested
against.


In [ ]:
# Package-API training run, equivalent to Section 5 below.
using PhasorNetworks, Lux

rng_pkg = Xoshiro(42)
n_in, n_hid, n_out = 4, 16, 2

chain = Chain(PhasorDense(n_in  => n_hid, normalize_to_unit_circle, use_bias=false),
              PhasorDense(n_hid => n_out, normalize_to_unit_circle, use_bias=false))
ps_pkg, st_pkg = Lux.setup(rng_pkg, chain)

# Phase target on the unit circle, complex-valued.
x_pkg = Phase.(2f0 .* rand(Xoshiro(1), Float32, n_in)  .- 1f0)
y_pkg = ComplexF32.(exp.(im .* π .* (2f0 .* rand(Xoshiro(2), Float32, n_out) .- 1f0)))

args_pkg = Args(lr=0.05, epochs=80)
losses_pkg, _, _ = ep_train(chain, ps_pkg, st_pkg, [(x_pkg, y_pkg)], args_pkg;
                            method=StaticEP(β=0.1f0, T_free=100, T_nudge=50))

println("Package-API EP training:")
println("  start cost = ", round(losses_pkg[1], digits=4))
println("  final cost = ", round(losses_pkg[end], digits=6))

plot(losses_pkg, lw=2, label="StaticEP(β=0.1)",
     xlabel="epoch", ylabel="cost  1 − sim(z_o, y)",
     title="Phasor EP via package API",
     yscale=:log10)


### Quick gradient sanity-check (package vs FD oracle)

The package's `fd_gradient_phasor` is the same coordinate-by-coordinate
finite-difference probe used in the existing demos. Comparing
`StaticEP` against it on the chain above confirms the integration
works end-to-end.


In [ ]:
# Compare per-layer EP gradient against the FD oracle for a sweep of β.
using LinearAlgebra: norm, dot

fd_pkg = fd_gradient_phasor(chain, ps_pkg, st_pkg, x_pkg, y_pkg; T=200)

println("  β     | layer_1 cos | layer_1 rel-err | layer_2 cos | layer_2 rel-err")
println("--------+-------------+-----------------+-------------+----------------")
for β_test in (0.001f0, 0.01f0, 0.1f0, 1.0f0)
    g, _ = ep_gradient(StaticEP(β=β_test, T_free=200, T_nudge=100),
                       chain, ps_pkg, st_pkg, x_pkg, y_pkg)
    cos1 = dot(vec(g.layer_1.weight), vec(fd_pkg.layer_1.weight)) /
           (norm(g.layer_1.weight) * norm(fd_pkg.layer_1.weight) + 1f-10)
    re1  = norm(g.layer_1.weight - fd_pkg.layer_1.weight) /
           norm(fd_pkg.layer_1.weight)
    cos2 = dot(vec(g.layer_2.weight), vec(fd_pkg.layer_2.weight)) /
           (norm(g.layer_2.weight) * norm(fd_pkg.layer_2.weight) + 1f-10)
    re2  = norm(g.layer_2.weight - fd_pkg.layer_2.weight) /
           norm(fd_pkg.layer_2.weight)
    println("  ", rpad(β_test, 6), "| ", rpad(round(cos1, digits=4), 12),
            "| ", rpad(round(re1, digits=4), 16),
            "| ", rpad(round(cos2, digits=4), 12),
            "| ", round(re2, digits=4))
end


### Lock-in EP via the package API

The package also ships `LockinEP` — the temporal real-probe lock-in gradient
extraction explored in Section 6 below. Same `ep_train`/`ep_gradient`
interface; just swap `StaticEP(...)` for `LockinEP(...)`. Lock-in
needs an adiabatic probe (small `ε`, slow `ω_p`) to match FD ground
truth, but it gets you the gradient from a single (free + lock-in)
settle pass instead of two (free + nudge).


In [ ]:
# Lock-in EP gradient comparison against the same FD oracle.
println("LockinEP vs FD (same chain / target as above):")
println("  ε      ω_p     | layer_1 cos | layer_1 rel-err | layer_2 cos | layer_2 rel-err")
println("------------------+-------------+-----------------+-------------+----------------")
for (ε_test, ω_test) in ((0.01f0, 0.01f0),
                         (0.05f0, 0.05f0),
                         (0.05f0, 0.10f0),
                         (0.10f0, 0.50f0))
    g, _ = ep_gradient(LockinEP(ε=ε_test, ω_p=ω_test,
                                n_cycles=8, T_warmup_cycles=2,
                                T_free=200, dt=0.1f0),
                       chain, ps_pkg, st_pkg, x_pkg, y_pkg)
    cos1 = dot(vec(g.layer_1.weight), vec(fd_pkg.layer_1.weight)) /
           (norm(g.layer_1.weight) * norm(fd_pkg.layer_1.weight) + 1f-10)
    re1  = norm(g.layer_1.weight - fd_pkg.layer_1.weight) /
           norm(fd_pkg.layer_1.weight)
    cos2 = dot(vec(g.layer_2.weight), vec(fd_pkg.layer_2.weight)) /
           (norm(g.layer_2.weight) * norm(fd_pkg.layer_2.weight) + 1f-10)
    re2  = norm(g.layer_2.weight - fd_pkg.layer_2.weight) /
           norm(fd_pkg.layer_2.weight)
    println("  ε=", rpad(ε_test, 5), " ω_p=", rpad(ω_test, 5),
            "| ", rpad(round(cos1, digits=4), 12),
            "| ", rpad(round(re1, digits=4), 16),
            "| ", rpad(round(cos2, digits=4), 12),
            "| ", round(re2, digits=4))
end


In [ ]:
# Side-by-side training: StaticEP vs LockinEP on the same fixed pattern.
rng_static = Xoshiro(42); rng_lockin = Xoshiro(42)
chain2 = Chain(PhasorDense(n_in  => n_hid, normalize_to_unit_circle, use_bias=false),
               PhasorDense(n_hid => n_out, normalize_to_unit_circle, use_bias=false))
ps_s, st_s = Lux.setup(rng_static, chain2)
ps_l, st_l = Lux.setup(rng_lockin, chain2)

losses_static, _, _ = ep_train(chain2, ps_s, st_s, [(x_pkg, y_pkg)], Args(lr=0.05, epochs=40);
                               method=StaticEP(β=0.1f0, T_free=100, T_nudge=50))
losses_lockin, _, _ = ep_train(chain2, ps_l, st_l, [(x_pkg, y_pkg)], Args(lr=0.05, epochs=40);
                               method=LockinEP(ε=0.05f0, ω_p=0.05f0,
                                               n_cycles=4, T_warmup_cycles=2,
                                               T_free=100, dt=0.1f0))

println("StaticEP final cost:  ", round(losses_static[end], digits=6))
println("LockinEP final cost:  ", round(losses_lockin[end], digits=6))

plot(losses_static, lw=2, label="StaticEP(β=0.1)",
     xlabel="epoch", ylabel="cost",
     title="StaticEP vs LockinEP (same target, same init)",
     yscale=:log10)
plot!(losses_lockin, lw=2, label="LockinEP(ε=0.05, ω_p=0.05)")


## 1. State, energy, and dynamics

A 2-layer phasor MLP has one hidden layer $z_h \in \mathbb{C}^{n_h}$
and one output $z_o \in \mathbb{C}^{n_o}$. All states are
unit-modulus complex; we maintain `z` as `ComplexF64` arrays and
project back to the unit circle after each update.

Weights $W_1, W_2$ are real-valued (so the recurrent feedback term
$W_2^\top$ — which EP needs for symmetric coupling — is
well-defined as ordinary matrix transpose). Inputs $x$ and targets
$y$ are unit-complex phase patterns.

The energy function is

$$\Phi(z_h, z_o; \theta, \beta) \;=\; \tfrac{1}{2}\,\text{Re}\langle z_h, K_h z_h\rangle \;+\; \tfrac{1}{2}\,\text{Re}\langle z_o, K_o z_o\rangle$$
$$\quad+\; \text{Re}\langle W_1 x, z_h \rangle \;+\; \text{Re}\langle W_2 z_h, z_o \rangle \;-\; \beta\,C(z_o, y)$$

with cost $C(z_o, y) = 1 - (1/d)\,\text{Re}\langle z_o, y \rangle$
(one minus cosine similarity).

Wirtinger gradients give the settling forces:

$$\partial \Phi / \partial \bar z_h \;=\; \tfrac{1}{2} K_h z_h + W_1 x + W_2^\top z_o$$
$$\partial \Phi / \partial \bar z_o \;=\; \tfrac{1}{2} K_o z_o + W_2 z_h + \beta\, y / (2 n_o)$$

The forward drive `W·z_prev` is the **VSA bundle** primitive
applied to the previous layer; the backward feedback `W^⊤·z_next`
is the symmetric coupling EP needs. The cost gradient $\beta\,y/(2n_o)$
is a *constant* pull toward the target $y$ in the complex plane —
no slope factor, no saturation derivative.

**One subtle convention point.** The Wirtinger gradient
$\partial C/\partial \bar z_o = -y/(2n_o)$ is "half" of what a
finite-difference probe of real parameters would measure: the
real-parameter chain rule picks up a factor of 2 when packaged
back into a complex update. To make the EP gradient match
finite-difference ground truth on real $W_1, W_2$, we use the
**real-parameter convention** in the settling code, with nudge
$\beta\,y/n_o$ (no factor of 2 in the denominator). This is the
same convention vanilla EP uses with MSE — there's no $\tfrac{1}{2}$
in the cost-derivative either, by construction.

For this first prototype we set $K = 0$ (no decay, no oscillation)
to isolate the phase-consensus dynamics. See the design doc for
how to add SSM dynamics back in.


In [ ]:
# Real-valued weights, complex unit-modulus states.
mutable struct PhasorEPNet
    W1::Matrix{Float64}   # (n_hid, n_in)
    W2::Matrix{Float64}   # (n_out, n_hid)
end

function PhasorEPNet(n_in, n_hid, n_out; rng=Xoshiro(42), scale=0.5)
    PhasorEPNet(randn(rng, n_hid, n_in) * scale,
                randn(rng, n_out, n_hid) * scale)
end

# Cost: 1 - cosine similarity. ∈ [0, 2], minimized at 0 when z_o = y.
phasor_cost(z_o, y) = 1.0 - real(dot(y, z_o)) / length(z_o)

# Project a complex vector onto the unit circle. With ε for numerical safety
# (so a zero drive maps to itself rather than NaN).
unit_project(z; ε=1e-12) = z ./ (abs.(z) .+ ε)

# Convert a real phase vector θ ∈ [-1, 1] into a unit complex vector.
phase_to_complex(θ) = exp.(im .* π .* θ)

# Random unit-complex pattern of length n.
random_phasor(n; rng=Xoshiro(0)) = phase_to_complex(2 .* rand(rng, n) .- 1)


In [ ]:
# Settle the network by descending −∂Φ/∂z̄ on each layer, with damped updates
# and unit-circle projection (the phasor analog of vanilla EP's tanh squashing).
#
# Equilibrium fixed-point conditions (β = 0):
#   z_h ∝ W1·x + W2'·z_o
#   z_o ∝ W2·z_h
# The damping `dt` is a stability knob; dt=1 is pure fixed-point iteration.
function phasor_settle!(net::PhasorEPNet, x, z_h, z_o, y, β; T=100, dt=0.5)
    d_o = length(z_o)
    for _ in 1:T
        # Wirtinger gradient w.r.t. z̄_h: forward drive + backward feedback.
        grad_h = net.W1 * x .+ net.W2' * z_o
        # Wirtinger gradient w.r.t. z̄_o: forward drive + nudge.
        grad_o = net.W2 * z_h .+ (β / d_o) .* y
        # Damped projected update — analogue of (1-dt)·h + dt·tanh(grad_h)
        # in vanilla EP, with normalize_to_unit_circle in place of tanh.
        z_h .= (1-dt) .* z_h .+ dt .* unit_project(grad_h)
        z_o .= (1-dt) .* z_o .+ dt .* unit_project(grad_o)
    end
end

# Recording variant — returns trajectories for visualization.
function phasor_settle_with_history(net::PhasorEPNet, x, z_h0, z_o0, y, β; T=100, dt=0.5)
    z_h = copy(z_h0); z_o = copy(z_o0)
    h_hist = [copy(z_h)]; o_hist = [copy(z_o)]
    d_o = length(z_o)
    for _ in 1:T
        grad_h = net.W1 * x .+ net.W2' * z_o
        grad_o = net.W2 * z_h .+ (β / d_o) .* y
        z_h .= (1-dt) .* z_h .+ dt .* unit_project(grad_h)
        z_o .= (1-dt) .* z_o .+ dt .* unit_project(grad_o)
        push!(h_hist, copy(z_h)); push!(o_hist, copy(z_o))
    end
    h_hist, o_hist
end

# Energy function — useful for sanity-checking the settling.
function phasor_energy(net::PhasorEPNet, x, z_h, z_o, y, β)
    coupling_in  = real(dot(z_h, net.W1 * x))
    coupling_hid = real(dot(z_o, net.W2 * z_h))
    coupling_in + coupling_hid - β * phasor_cost(z_o, y)
end


## 2. Free-phase settling

With $\beta = 0$ the network settles to a "phase-consensus"
equilibrium determined entirely by the input pattern $x$ and the
recurrent coupling between layers. We visualize the convergence in
both magnitude (which should rise from ~0 to ~1 as the unit-circle
projection takes effect) and angle (which converges to the
equilibrium phase pattern).


In [ ]:
rng = Xoshiro(42)
n_in, n_hid, n_out = 4, 16, 2

net = PhasorEPNet(n_in, n_hid, n_out; rng=rng, scale=0.4)
x   = random_phasor(n_in; rng=Xoshiro(1))   # input phase pattern
y   = random_phasor(n_out; rng=Xoshiro(2))  # target phase pattern

# Initialize at zero (a "neutral" point at the origin of ℂ).
z_h0 = zeros(ComplexF64, n_hid)
z_o0 = zeros(ComplexF64, n_out)

T_settle = 100
h_hist, o_hist = phasor_settle_with_history(net, x, z_h0, z_o0, y, 0.0; T=T_settle)

H = hcat(h_hist...)   # (n_hid, T+1)
O = hcat(o_hist...)   # (n_out, T+1)

# Energy trajectory (recompute from the saved states).
energies = [phasor_energy(net, x, h_hist[t], o_hist[t], y, 0.0) for t in 1:length(h_hist)]

println("Free equilibrium reached after $T_settle steps.")
println("  |z_o|     = ", round.(abs.(O[:, end]), digits=4))
println("  angle(z_o)= ", round.(angle.(O[:, end]) ./ π, digits=4), " (units of π)")
println("  cost C(z_o, y) = ", round(phasor_cost(O[:, end], y), digits=4))
println("  energy Φ       = ", round(energies[end], digits=4))


In [ ]:
ch = [1, 4, 7, 10, 13]   # five representative hidden channels

p_mag = plot(xlabel="settling step", ylabel="|state|",
             title="Free-phase magnitude convergence", legend=:bottomright)
for i in ch
    plot!(p_mag, abs.(H[i, :]), lw=1.2, alpha=0.7, label="z_h$i")
end
plot!(p_mag, abs.(O[1, :]), lw=2.5, color=:black,    label="z_o[1]")
plot!(p_mag, abs.(O[2, :]), lw=2.5, color=:firebrick, label="z_o[2]")

p_ang = plot(xlabel="settling step", ylabel="angle(state) / π",
             title="Free-phase angle convergence", legend=:right,
             ylim=(-1.05, 1.05))
for i in ch
    plot!(p_ang, angle.(H[i, :]) ./ π, lw=1.2, alpha=0.7, label="z_h$i")
end
plot!(p_ang, angle.(O[1, :]) ./ π, lw=2.5, color=:black,    label="z_o[1]")
plot!(p_ang, angle.(O[2, :]) ./ π, lw=2.5, color=:firebrick, label="z_o[2]")

p_eng = plot(energies, xlabel="settling step", ylabel="Φ",
             title="Energy trajectory (β=0)", lw=2, legend=false)
hline!(p_eng, [energies[end]], ls=:dash, color=:gray)

plot(p_mag, p_ang, p_eng, layout=(3, 1), size=(800, 900))


## 3. Nudged phase and the EP gradient

With $\beta > 0$ the cost gradient adds a constant force $\beta\,y/(2n_o)$
on $z_o$, pulling it toward the target. We start the nudged phase
from the free equilibrium so the perturbation is small and the
gradient extraction is well-conditioned.

The Hebbian update for $W_1$ at an equilibrium is the complex outer
product
$$\text{hebb}_{W_1} = \text{Re}\bigl(z_h \cdot x^*\bigr)$$
(the local gradient $\partial \Phi / \partial W_1$ evaluated at the
equilibrium states). The EP gradient estimate is the difference
$(\text{hebb}_\text{nudge} - \text{hebb}_\text{free})/\beta$, with
a sign flip because $\Phi$ contains $-\beta C$ and we want
$dL/dW = -d\Phi/dW$.


In [ ]:
# Hebbian outer products at given equilibrium states.
hebb_W1(x, z_h) = real.(z_h * adjoint(x))    # (n_hid, n_in)
hebb_W2(z_h, z_o) = real.(z_o * adjoint(z_h)) # (n_out, n_hid)

function phasor_ep_gradient(net::PhasorEPNet, x, y, β;
                            T_free=100, T_nudge=50, dt=0.5)
    z_h = zeros(ComplexF64, size(net.W1, 1))
    z_o = zeros(ComplexF64, size(net.W2, 1))
    phasor_settle!(net, x, z_h, z_o, y, 0.0; T=T_free, dt=dt)
    h1_free = hebb_W1(x, z_h)
    h2_free = hebb_W2(z_h, z_o)

    z_h_n = copy(z_h); z_o_n = copy(z_o)
    phasor_settle!(net, x, z_h_n, z_o_n, y, β; T=T_nudge, dt=dt)
    h1_nudge = hebb_W1(x, z_h_n)
    h2_nudge = hebb_W2(z_h_n, z_o_n)

    # Sign flip: Φ has −β·C, we want dL/dW = −dΦ/dW.
    grad_W1 = -(h1_nudge - h1_free) / β
    grad_W2 = -(h2_nudge - h2_free) / β
    return grad_W1, grad_W2, z_h, z_o
end

g_W1, g_W2, z_h_free, z_o_free = phasor_ep_gradient(net, x, y, 0.1)
println("EP gradient at β=0.1:")
println("  ‖∇W1‖ = ", round(norm(g_W1), digits=6))
println("  ‖∇W2‖ = ", round(norm(g_W2), digits=6))
println("  free cost = ", round(phasor_cost(z_o_free, y), digits=4))


## 4. Gradient verification vs. finite differences

We use the same protocol as `ep_demo.ipynb`: compute the FD
ground-truth gradient of $L(W_1) = C(z_o^*_{\text{free}}(W_1), y)$ by
perturbing each entry of $W_1$, re-running the free-phase settle, and
forward-differencing the resulting cost. We then compare the EP
estimate at various $\beta$ values, reporting **all three** metrics:

* **cosine similarity** — direction-only, scale-invariant
* **relative error** $\|\text{est} - \text{fd}\|/\|\text{fd}\|$ — direction + magnitude
* **relative magnitude error** $\bigl|\,\|\text{est}\| - \|\text{fd}\|\,\bigr|/\|\text{fd}\|$ — magnitude only

As we found in the EP/hEP demos, cosine alone is misleading. The
relative-error curve reveals whether the EP estimate has both the
right direction *and* the right magnitude, which is what actually
determines whether SGD using it converges at the intended rate.


In [ ]:
# Finite-difference ground truth: dL/dW1 where L = phasor_cost at the free equilibrium.
function fd_grad_W1(net::PhasorEPNet, x, y; ε=1e-5, T=100, dt=0.5)
    function loss_at(W1)
        np = PhasorEPNet(W1, net.W2)
        z_h = zeros(ComplexF64, size(W1, 1))
        z_o = zeros(ComplexF64, size(net.W2, 1))
        phasor_settle!(np, x, z_h, z_o, y, 0.0; T=T, dt=dt)
        phasor_cost(z_o, y)
    end
    g = zeros(size(net.W1)); base = loss_at(net.W1)
    for i in eachindex(net.W1)
        Wp = copy(net.W1); Wp[i] += ε
        g[i] = (loss_at(Wp) - base) / ε
    end
    g
end

fd = fd_grad_W1(net, x, y; T=100)
fd_norm = norm(fd)
println("FD gradient ‖∇W1‖ = ", round(fd_norm, digits=6))


In [ ]:
function compare_to_fd(g, fd, fd_norm)
    cs = dot(vec(g), vec(fd)) / (norm(g) * fd_norm + 1e-10)
    rel_err = norm(g - fd) / fd_norm
    rel_mag = abs(norm(g) - fd_norm) / fd_norm
    cs, rel_err, rel_mag
end

βs = [0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0]
ep_cosines, ep_rel_errs, ep_rel_mag_errs = Float64[], Float64[], Float64[]
ep_norms = Float64[]
for β in βs
    g_W1, _, _, _ = phasor_ep_gradient(net, x, y, β; T_free=100, T_nudge=50)
    cs, re, rm = compare_to_fd(g_W1, fd, fd_norm)
    push!(ep_cosines, cs); push!(ep_rel_errs, re); push!(ep_rel_mag_errs, rm)
    push!(ep_norms, norm(g_W1))
end

println("\n  β    | cosine | rel-err | rel-mag-err | EP norm  | FD norm")
println("-------+--------+---------+-------------+----------+---------")
for (β, c, re, rm, en) in zip(βs, ep_cosines, ep_rel_errs, ep_rel_mag_errs, ep_norms)
    println("  ", rpad(β, 5), "| ", rpad(round(c, digits=4), 7),
            "| ", rpad(round(re, digits=4), 8),
            "| ", rpad(round(rm, digits=4), 12),
            "| ", rpad(round(en, digits=5), 9),
            "| ", round(fd_norm, digits=5))
end


In [ ]:
p_cos = plot(βs, ep_cosines, xlabel="β", ylabel="Cosine similarity",
             title="Direction quality (scale-invariant)",
             xscale=:log10, marker=:circle, lw=2, color=:steelblue,
             label="phasor EP", ylim=(-0.2, 1.1))
hline!(p_cos, [1.0], ls=:dash, color=:gray, label="perfect")
hline!(p_cos, [0.0], ls=:dot,  color=:red,  label="random")

p_err = plot(βs, ep_rel_errs, xlabel="β", ylabel="‖est − fd‖ / ‖fd‖",
             title="Total relative error (direction + magnitude)",
             xscale=:log10, yscale=:log10, marker=:diamond, lw=2,
             color=:steelblue, label="rel-err (mag + dir)")
plot!(p_err, βs, ep_rel_mag_errs, lw=1.5, ls=:dash, marker=:utriangle,
      color=:firebrick, label="rel-err (mag only)")

plot(p_cos, p_err, layout=(2, 1), size=(800, 700))


## 5. A simple training run

To check that the EP gradient is actually useful for learning, we
train the same 2-layer phasor network to map a fixed input $x$ to a
fixed target $y$. With a single training example this is a
trivial task — the loss should drive cleanly to zero — but it
exercises the full EP loop end-to-end (free settle, nudged settle,
Hebbian outer product, parameter update).

We compare two $\beta$ values: a small one (low bias, slow updates)
and a moderate one (faster updates, larger systematic error). The
loss curves will show the usual phasor-EP trade-off — small $\beta$
gives smoother descent at the cost of step magnitude.


In [ ]:
function phasor_ep_train!(net::PhasorEPNet, x, y;
                          β=0.1, lr=0.05, epochs=80, T_free=100, T_nudge=50)
    losses = Float64[]
    for epoch in 1:epochs
        g_W1, g_W2, z_h_free, z_o_free = phasor_ep_gradient(net, x, y, β;
                                                             T_free=T_free,
                                                             T_nudge=T_nudge)
        push!(losses, phasor_cost(z_o_free, y))
        net.W1 .-= lr .* g_W1
        net.W2 .-= lr .* g_W2
    end
    losses
end

# Two networks with identical init, two different β values.
function fresh_net()
    PhasorEPNet(n_in, n_hid, n_out; rng=Xoshiro(42), scale=0.4)
end

net_lo = fresh_net()
net_hi = fresh_net()

losses_lo = phasor_ep_train!(net_lo, x, y; β=0.05, lr=0.05, epochs=80)
losses_hi = phasor_ep_train!(net_hi, x, y; β=0.5,  lr=0.05, epochs=80)

println("Final cost (β=0.05): ", round(losses_lo[end], digits=6))
println("Final cost (β=0.5):  ", round(losses_hi[end], digits=6))

plot(losses_lo, lw=2, label="β = 0.05 (low bias)",
     xlabel="epoch", ylabel="cost  1 − sim(z_o, y)",
     title="Phasor EP — fixed-pattern association",
     yscale=:log10, legend=:topright)
plot!(losses_hi, lw=2, label="β = 0.5 (moderate bias)")


## 6. Lock-in detection (temporal real probe)

The hEP/Cauchy trick can be reformulated in *time* instead of *space* —
drive the nudge as a small high-frequency oscillation
$\beta(t) = \epsilon\,\cos(\omega_p t)$, then **lock-in detect**
the network's response by multiplying the recorded Hebbian outer
product by $e^{-i\omega_p t}$ and time-averaging over an integer
number of probe cycles. This extracts the linear-response coefficient
$d\,\text{hebb}/d\beta$ at $\beta = 0$, which by the EP gradient
theorem is (minus) the loss gradient.

**Why a real probe, not a complex one?** With phasor EP's `unit_project`
activation, the equilibrium $z^*(\beta, \bar\beta)$ depends on both
$\beta$ and $\bar\beta$ (it's not holomorphic in $\beta$). A
complex probe $\epsilon\,e^{i\omega_p t}$ would extract only the
Wirtinger $\partial z^* / \partial \beta$, missing the
$\partial z^* / \partial \bar\beta$ part — wrong gradient on
real-valued weights. A real cosine probe excites both sidebands; the
$+\omega_p$ Fourier component picks up the full
$d/d\beta_{\rm real} = \partial/\partial\beta + \partial/\partial\bar\beta$,
which is what FD measures. See `docs/phasor_ep_design.md` for the
full derivation.

**Implementation specifics for stability:**

1. **DC subtraction** — subtract the free-equilibrium Hebbian before
   demodulating, otherwise DC leakage from imperfect cycle counting
   dominates the estimate.
2. **Integer probe cycles** — choose `T_lockin = n_cycles · period_steps`
   exactly so even-harmonic terms cancel.
3. **Warm-up** — discard the first ~2 probe cycles so the equilibrium
   has caught up to the modulation (transient response decays).
4. **Adiabatic regime** — keep $\omega_p$ slow enough that the
   network can track the probe quasi-statically (well below the
   relaxation rate set by the damping `dt` and the coupling
   strength).


In [ ]:
# Lock-in EP gradient. The lock-in coefficient at +ω_p of a real probe of
# amplitude ε is (ε/2)·d/dβ_real(observable), so the EP gradient is
#   −2·Re(H_+) / ε.
function phasor_ep_lockin(net::PhasorEPNet, x, y;
                          ε=0.05, ω_p=0.05, n_cycles=8,
                          T_free=400, T_warmup_cycles=2, dt=0.1)
    n_hid = size(net.W1, 1); n_out = size(net.W2, 1)
    d_o = n_out

    # Free settle to the β=0 equilibrium.
    z_h = zeros(ComplexF64, n_hid); z_o = zeros(ComplexF64, n_out)
    phasor_settle!(net, x, z_h, z_o, y, 0.0; T=T_free, dt=dt)
    H1_dc = z_h * adjoint(x)         # DC hebb to subtract before demod
    H2_dc = z_o * adjoint(z_h)

    period_steps = round(Int, 2π / (ω_p * dt))

    # Warm-up phase: drive the probe but don't accumulate (transients die).
    for t in 1:(T_warmup_cycles * period_steps)
        β_t = ε * cos(ω_p * t * dt)
        grad_h = net.W1 * x .+ net.W2' * z_o
        grad_o = net.W2 * z_h .+ (β_t / d_o) .* y
        z_h .= (1-dt) .* z_h .+ dt .* unit_project(grad_h)
        z_o .= (1-dt) .* z_o .+ dt .* unit_project(grad_o)
    end

    # Integration phase: n_cycles complete probe periods.
    H1_acc = zeros(ComplexF64, size(net.W1))
    H2_acc = zeros(ComplexF64, size(net.W2))
    T_lockin = n_cycles * period_steps
    for t in 1:T_lockin
        β_t = ε * cos(ω_p * t * dt)
        grad_h = net.W1 * x .+ net.W2' * z_o
        grad_o = net.W2 * z_h .+ (β_t / d_o) .* y
        z_h .= (1-dt) .* z_h .+ dt .* unit_project(grad_h)
        z_o .= (1-dt) .* z_o .+ dt .* unit_project(grad_o)
        # DC-subtracted hebb, demodulated at the probe frequency.
        demod = exp(-im * ω_p * t * dt)
        H1_acc .+= ((z_h * adjoint(x))   .- H1_dc) .* demod
        H2_acc .+= ((z_o * adjoint(z_h)) .- H2_dc) .* demod
    end

    grad_W1 = -2 .* real.(H1_acc) ./ (T_lockin * ε)
    grad_W2 = -2 .* real.(H2_acc) ./ (T_lockin * ε)
    return grad_W1, grad_W2
end

# Sanity check at one operating point.
g_lockin, _ = phasor_ep_lockin(net, x, y; ε=0.01, ω_p=0.01,
                                n_cycles=8, T_free=400, dt=0.1)
cs_lk, re_lk, rm_lk = compare_to_fd(g_lockin, fd, fd_norm)
println("Lock-in EP at (ε=0.01, ω_p=0.01):")
println("  cosine     = ", round(cs_lk, digits=4))
println("  rel-err    = ", round(re_lk, digits=4))
println("  rel-mag    = ", round(rm_lk, digits=4))
println("  ‖∇W1‖     = ", round(norm(g_lockin), digits=5))
println("  ‖fd‖      = ", round(fd_norm, digits=5))


### 6.1 The (ε, ω_p) trade-off

Same heatmap idea as the hEP `(N, r)` sweep — vary the two probe knobs
and visualize where lock-in matches FD. Expected structure:

* **Small ε, slow ω_p**: deep adiabatic, exact gradient — the "good basin."
* **Large ε**: nonlinear contamination O(ε²) bias.
* **Fast ω_p**: non-adiabatic, the network can't track the probe and
  the magnitude is wrong (still directionally OK for a while).
* **Cliff at very fast ω_p** ≈ relaxation rate.

The vertical resolution (`n_ω`) and horizontal resolution (`n_ε`) are
the only knobs you need to tune to refine.


In [ ]:
# 2D sweep over (ε, ω_p) for the lock-in gradient.
# Cost per cell: one settle of T_free + (T_warmup_cycles + n_cycles)·period_steps.
# At small ω_p the period is long, so total work scales like 1/ω_p — slow probes
# are expensive. Default grid keeps total time under ~30 s.
n_ε  = 8
n_ωp = 12
ε_lo, ε_hi  = 1e-3, 1.0
ωp_lo, ωp_hi = 5e-3, 1.0

ε_grid  = exp10.(range(log10(ε_lo),  log10(ε_hi);  length=n_ε))
ωp_grid = exp10.(range(log10(ωp_lo), log10(ωp_hi); length=n_ωp))

err_grid = fill(NaN, length(ωp_grid), length(ε_grid))
@time for (i, ωp) in enumerate(ωp_grid), (j, ε) in enumerate(ε_grid)
    try
        g, _ = phasor_ep_lockin(net, x, y; ε=ε, ω_p=ωp,
                                n_cycles=4, T_free=200, dt=0.1)
        e = norm(g - fd) / fd_norm
        err_grid[i, j] = isfinite(e) ? e : NaN
    catch
        err_grid[i, j] = NaN
    end
end

best_idx = argmin(replace(err_grid, NaN => Inf))
best_ωp  = ωp_grid[best_idx[1]]
best_ε   = ε_grid[best_idx[2]]
best_err = err_grid[best_idx]
println("Best (ε, ω_p) = (", round(best_ε, sigdigits=3), ", ",
        round(best_ωp, sigdigits=3), ")  rel-err = ",
        round(best_err, sigdigits=4))

lo, hi = 1e-4, 10.0
logerr = log10.(clamp.(err_grid, lo, hi))
h = heatmap(ε_grid, ωp_grid, logerr;
            xlabel="ε (probe amplitude)",
            ylabel="ω_p (probe frequency)",
            xscale=:log10, yscale=:log10,
            title="Lock-in EP error  log10(‖est − fd‖ / ‖fd‖)",
            color=:viridis, colorbar_title="log10(rel-err)",
            clim=(log10(lo), log10(hi)))
scatter!(h, [best_ε], [best_ωp]; marker=:star5, ms=10, color=:white,
         markerstrokecolor=:black,
         label="best (ε=$(round(best_ε,sigdigits=3)), ω_p=$(round(best_ωp,sigdigits=3)))")
h


### 6.2 Comparison with static EP

Putting both methods on one figure makes the trade-off concrete:

* **Static EP** has direction quality but magnitude bias growing as
  $O(\beta)$ — visible as `rel-err ≈ β` curve.
* **Lock-in EP** can hit the FD noise floor across many `ω_p` values
  *if* `ε` is small enough — visible as a flat error floor.

Both are vulnerable to a "regime cliff": static EP at large $\beta$
where the nudge dominates the dynamics; lock-in at fast $\omega_p$
where the network can't track.


In [ ]:
# Side-by-side: static EP vs lock-in EP, both as 1-D sweeps.
# Static EP: sweep β.
# Lock-in EP: sweep ε at a chosen "good" ω_p; sweep ω_p at a chosen "good" ε.
ωp_fixed = 0.01    # slow probe — deep adiabatic
ε_fixed  = 0.01    # small probe — linear regime

# Static EP across β
β_grid = exp10.(range(-3, 0.3; length=10))
static_rel = Float64[]
for β in β_grid
    g, _, _, _ = phasor_ep_gradient(net, x, y, β; T_free=100, T_nudge=50)
    push!(static_rel, norm(g - fd) / fd_norm)
end

# Lock-in EP across ε at fixed ω_p
ε_sweep = exp10.(range(-3, 0; length=8))
lockin_rel_ε = Float64[]
for ε in ε_sweep
    g, _ = phasor_ep_lockin(net, x, y; ε=ε, ω_p=ωp_fixed,
                            n_cycles=4, T_free=200, dt=0.1)
    push!(lockin_rel_ε, norm(g - fd) / fd_norm)
end

# Lock-in EP across ω_p at fixed ε
ωp_sweep = exp10.(range(-2.5, 0; length=10))
lockin_rel_ωp = Float64[]
for ωp in ωp_sweep
    g, _ = phasor_ep_lockin(net, x, y; ε=ε_fixed, ω_p=ωp,
                            n_cycles=4, T_free=200, dt=0.1)
    push!(lockin_rel_ωp, norm(g - fd) / fd_norm)
end

p1 = plot(β_grid, static_rel,
          xlabel="β (static EP) — or ε (lock-in)",
          ylabel="‖est − fd‖ / ‖fd‖",
          title="Static EP β-sweep vs Lock-in ε-sweep  (ω_p = $ωp_fixed)",
          xscale=:log10, yscale=:log10,
          marker=:circle, lw=2, color=:red, label="Static EP (vary β)")
plot!(p1, ε_sweep, lockin_rel_ε, marker=:diamond, lw=2, color=:steelblue,
      label="Lock-in EP (vary ε)")

p2 = plot(ωp_sweep, lockin_rel_ωp,
          xlabel="ω_p (probe frequency)", ylabel="‖est − fd‖ / ‖fd‖",
          title="Lock-in EP ω_p-sweep  (ε = $ε_fixed)",
          xscale=:log10, yscale=:log10,
          marker=:square, lw=2, color=:darkorange, label="Lock-in rel-err")

plot(p1, p2, layout=(2, 1), size=(800, 700))


## Notes and next steps

This prototype is intentionally minimal:

* `K = 0` — no SSM dynamics. The next step is to add per-channel
  decay $\lambda$ and oscillation $\omega$ and check that the
  EP gradient still tracks FD when the self-energy term
  $\tfrac{1}{2}\text{Re}\langle z, K z \rangle$ is included.
* **Real-valued weights.** Conjugate-transpose feedback collapses to
  ordinary transpose. With complex weights you get the more general
  Hermitian condition.
* **Single training example.** Multi-example batching is a trivial
  loop wrapper; the more interesting extension is using a `Codebook`
  readout with cross-entropy as in `src/hep.jl`.
* **Discrete-time settling.** Same fixed-point iteration as vanilla
  EP. A continuous-time port would integrate the ODE
  $dz/dt = -\partial \Phi/\partial \bar z$ with a stiff solver and
  is a natural successor experiment.

The headline result is the same as vanilla EP: cosine similarity
stays near 1 across many $\beta$ values, but the relative-error
metric exposes the magnitude bias that grows linearly with $\beta$.
That's the regime where holomorphic EP pays off — but here, in the
phasor-EP setting, we have an open question: can the
contour-integration trick from hEP be applied to the *real* nudge
parameter $\beta$ here, or do we need to reformulate it through the
phase angle of a complex modulator? See `docs/phasor_ep_design.md`
for further discussion.
